In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval



In [21]:
# =========================
# PARÂMETROS
# =========================
ticker = "^BVSP" # código do ativo no yahoo
start_date = "2000-01-01"
end_date = None

df = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    progress=False,
    auto_adjust=True,
    back_adjust=True,
    multi_level_index=False,
    
)
df['ret'] = df['Close'].pct_change()
df.dropna(inplace=True)
df


,Close,High,Low,Open,Volume,ret
Date,,,,,,
2000-01-04,15851.0,16908.0,15851.0,16908.0,0,-0.063733
2000-01-05,16245.0,16302.0,15350.0,15871.0,0,0.024856
2000-01-06,16107.0,16499.0,15977.0,16237.0,0,-0.008495
2000-01-07,16309.0,16449.0,16125.0,16125.0,0,0.012541
2000-01-10,17022.0,17057.0,16325.0,16325.0,0,0.043718
...,...,...,...,...,...,...
2025-12-15,162482.0,163073.0,160766.0,160766.0,8229000,0.010674
2025-12-16,158578.0,162482.0,158558.0,162482.0,9920300,-0.024027
2025-12-17,157327.0,158611.0,156351.0,158578.0,11344300,-0.007889


In [22]:
import ipeadatapy as ip
cdi = (
    ip.timeseries('SGS366_CDI366')
    .rename(columns = {'VALUE ((%))' : 'cdi'})[['cdi']]
)
cdi = cdi.reindex(df.index).dropna()
cdi

,cdi
Date,
2000-01-04,0.068218
2000-01-05,0.068184
2000-01-06,0.068218
2000-01-07,0.068218
2000-01-10,0.068218
...,...
2025-12-12,0.055131
2025-12-15,0.055131
2025-12-16,0.055131


In [30]:
custo = 0.01/100
df['cdi'] = cdi/100
df["position"] = 0

is_first_day = df.index.to_series().dt.is_month_start
is_last_day  = df.index.to_series().dt.is_month_end

df.loc[is_first_day | is_last_day, "position"] = 1


df["strategy_ret"] = np.where((df.loc[: , "position"].fillna(False) == 1) 
                                        , df.loc[: , "ret"] - custo
                                        , np.where((df.loc[: , "position"].fillna(False) == 0)
                                                    , df.loc[: , "cdi"]
                                                    , 0 ),
                                            )

df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.dropna(inplace=True)
df

,Close,High,Low,Open,Volume,ret,cdi,position,strategy_ret,strategy,buy_hold
Date,,,,,,,,,,,
2000-01-04,15851.0,16908.0,15851.0,16908.0,0,-0.063733,0.000682,0,0.000682,0.000682,-0.063733
2000-01-05,16245.0,16302.0,15350.0,15871.0,0,0.024856,0.000682,0,0.000682,0.001364,-0.038877
2000-01-06,16107.0,16499.0,15977.0,16237.0,0,-0.008495,0.000682,0,0.000682,0.002046,-0.047371
2000-01-07,16309.0,16449.0,16125.0,16125.0,0,0.012541,0.000682,0,0.000682,0.002728,-0.034830
2000-01-10,17022.0,17057.0,16325.0,16325.0,0,0.043718,0.000682,0,0.000682,0.003411,0.008888
...,...,...,...,...,...,...,...,...,...,...,...
2025-12-12,160766.0,161263.0,159189.0,159189.0,7671200,0.009906,0.000551,0,0.000551,3.606454,3.183289
2025-12-15,162482.0,163073.0,160766.0,160766.0,8229000,0.010674,0.000551,0,0.000551,3.607006,3.193963
2025-12-16,158578.0,162482.0,158558.0,162482.0,9920300,-0.024027,0.000551,0,0.000551,3.607557,3.169936


In [31]:
# =========================
# PLOT
# =========================
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}]]
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["cdi"].cumsum() * 100,
        name="cdi",
        line=dict(width=2)
    ),
    secondary_y=False
)


# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2)
    ),
    secondary_y=False
)


mask = df["position"].shift(1) == 1

fig.add_trace(
    go.Scatter(
        x=df.index[mask],
        y=df.loc[mask, "buy_hold"]*100,
        mode="markers",
        name="Buy Signal",
        marker=dict(
            symbol="triangle-up",
            size=10
        )
    ),
    secondary_y=False
)
# Layout
fig.update_layout(
    title = (
        f"{ticker} |primeiro e ultimo dia do mês "
        ),
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis2_title="Position",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    height=600
)

# Ajuste do eixo secundário
fig.update_yaxes(range=[-0.05, 1.05], secondary_y=True)

fig.show()



In [32]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    (df["strategy_ret"].mean() - df['cdi'].mean()) / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    (df["ret"].mean() - df['cdi'].mean()) / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 316.58%
Strategy Return:   360.87%

Buy & Hold Vol: 27.00%
Strategy Vol:   6.18%

Buy & Hold Sharpe: 0.03
Strategy Sharpe:   0.43
